# Track B — Qwen2.5-VL-7B (Colab A100) — **_0707 라운드**

전 라운드 `_0706`(7B, 생성 TTA4+투표)는 **LB 0.81** (val EM 0.5173, 1등 0.92). 로컬 재분석(`reports/track_b_0707.md`) 결론:
- no_ordering은 7B가 identity 투표로 사실상 해소 (em_no_ordering 0.596) — LL(tta1)은 투표에 전면 열세, E2(tta1) 제출 가치 없음
- 남은 병목 = **em_orderable 0.5025, 저합의 구간** → 이번 라운드 레버:

| 레버 | 셀 | 예상 비용 (A100) |
|---|---|---|
| **G1. 생성 TTA 4→8 확장** (기존 raw에 4뷰 추가) | G1a→G1b | val ~40분 + test ~35분 |
| **G2. 재학습**: no_ordering 오버샘플 ×2 + 3 epochs | B1→B4→C1→C2 | ~1.9× (_0706 학습 시간) |
| G3. (선택) LL 뷰 4 확장 재평가 | E1b→E2 | val ~3h |
| G4. 3090 24h 예산 실측 (규정 잔여 검증) | G4 | ~10분 |

픽셀 캡 OOM은 **근본 수정됨** (silent no-op → `cap_pixels` PIL 강제 캡, `tests/test_pixel_caps.py`).

사전 준비 (Drive `MyDrive/snuai/`): `snuai_code_0707.zip` (로컬 `python -m cloud.pack_code` 산출물 개명), `access_token`, `_0706` 산출물 유지 (`raw_val/raw_test_0706.jsonl`, `qwen25vl7b_merged_0706`).

실행 순서: A1→A2→A3→A4(재시작)→A1·A3 → **G1a→G1b(제출 #1 후보)** → B1→B2→B3→**B4(필수)**→C1→C2(제출 #2 후보) → (선택) E1b→로컬 fuse sweep→E2 → G4

**제출 규율(1일 2회)**: 판정 기준은 val EM **0.5173**(_0706 7B). 이를 못 넘는 구성은 제출하지 않는다.


In [ ]:
# A1) Drive 마운트 + 코드 압축 해제 (기존 코드 청소 후 — 구버전 파일 잔류 방지)
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/code && unzip -qo /content/drive/MyDrive/snuai/snuai_code_0707.zip -d /content/code
import sys; sys.path.insert(0, '/content/code')

In [ ]:
# A2) 대회 데이터 — Kaggle API v2 인증 (단일 토큰 방식)
# 토큰 발급: kaggle.com/settings → API → "Generate New Token" → 화면에 뜬 문자열 복사
# (구식 kaggle.json은 2026 API 개편으로 401 거부됨)
SLUG = 'snuaichallenge'
!pip install -qU kaggle  # v2 CLI 보장 (access_token 지원)

# 방법 1(권장): 토큰 문자열을 확장자 없는 파일 'access_token'으로 저장해 Drive의 snuai/에 업로드
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/snuai/access_token ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
# 방법 1이 번거로우면: 아래 두 줄 주석 해제 후 토큰을 직접 붙여넣기 (노트북 공유 금지!)
# TOKEN = '여기에-토큰-붙여넣기'
# !mkdir -p ~/.kaggle && printf %s {TOKEN} > ~/.kaggle/access_token

!kaggle competitions download {SLUG} -p /content
!unzip -qo /content/{SLUG}.zip -d /content/data
# API가 계속 거부되면 최후 수단: Drive에 올린 snuaichallenge.zip 사용
# !unzip -qo /content/drive/MyDrive/snuai/snuaichallenge.zip -d /content/data
!ls /content/data

In [ ]:
# A3) 경로 설정 — train.csv 위치 자동 탐지 (중첩 폴더 대응)
import os, glob, shutil
hits = sorted(glob.glob('/content/data/train.csv')
              + glob.glob('/content/data/*/train.csv')
              + glob.glob('/content/data/*/*/train.csv'))
assert hits, 'train.csv를 찾지 못함 — A2의 압축 해제 결과 확인'
DATA_ROOT = os.path.dirname(hits[0])
print('DATA_ROOT:', DATA_ROOT)

yaml_text = '\n'.join([
    f'data_dir: {DATA_ROOT}',
    f'train_csv: {DATA_ROOT}/train.csv',
    f'test_csv: {DATA_ROOT}/test.csv',
    f'sample_submission: {DATA_ROOT}/sample_submission.csv',
    f'train_image_dir: {DATA_ROOT}/train',
    f'test_image_dir: {DATA_ROOT}/test',
    'models_dir: /content/models',
    'outputs_dir: /content/outputs',
    'reports_dir: /content/reports',
])
open('/content/paths.yaml', 'w').write(yaml_text)
os.environ['SNUAI_PATHS_CONFIG'] = '/content/paths.yaml'
os.makedirs('/content/outputs', exist_ok=True)
# split.csv는 코드 번들에 있으므로 outputs_dir로 복사 (--fold val 경로가 참조)
shutil.copy('/content/code/outputs/split.csv', '/content/outputs/split.csv')

from src.data.loader import load_split
print('train rows:', len(load_split('train')), '| test rows:', len(load_split('test')))

In [ ]:
# A4) 설치 + 재현성 증빙
# ⚠️ 이 셀을 처음 실행한 뒤에는 반드시 [런타임 → 세션 다시 시작] 후 A1, A3만 재실행하고 B1로.
#    (unsloth가 pyarrow 등을 교체하므로 재시작 없이는 바이너리 불일치 에러 발생)
!pip install -q unsloth imagehash
import subprocess, sys
open('/content/drive/MyDrive/snuai/pip_freeze_colab.txt','w').write(
    subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True).stdout)
print('설치 완료 — 런타임을 재시작한 뒤 A1, A3 재실행 후 B1을 진행하세요.')

In [ ]:
# D0) [완료 2026-07-05 — 재실행 불필요, 결과는 reports/val_gap.md] [Phase 0 진단] val↔LB 괴리 — _0705 raw 투표 통계 비교 (GPU·unsloth 불필요, A1·A3만 실행 후 가능)
# 핵심 출력: "projected_lb" = val의 P(정답|합의도)를 test 합의도 분포에 적용한 예상 LB.
#   → 실제 LB 0.71에 근접하면 괴리는 "test가 val보다 쉬운 분포"로 해명 (지표·버그 아님).
# 유사도 표: corr(nn_train_sim, em) > 0 이 뚜렷하면 그룹 분리 split이 val을 구조적으로 어렵게 만든 것.
# 결과는 reports/val_gap.md에 기록할 것.
!cd /content/code && python -m src.eval.vote_stats \
    --raw-val  /content/drive/MyDrive/snuai/raw_val_0705.jsonl \
    --raw-test /content/drive/MyDrive/snuai/raw_test_0705.jsonl \
    --json /content/drive/MyDrive/snuai/vote_stats_0705.json

In [ ]:
# D1) [완료 2026-07-05 — 재실행 불필요, 결과는 reports/val_gap.md] [Phase 0 진단] 암기 체크 — _0705 3B 모델로 train fold 500샘플 EM (A100 ~10분, GPU 필요)
# train EM이 매우 높으면(>0.9) "test에 train 근접 중복이 남아 LB가 부풀 수 있다" 가설 지지.
# tta 1은 표가 1장뿐이라 분산 게이트가 전 샘플을 identity로 만들므로 반드시 게이트 off.
OLD = '/content/drive/MyDrive/snuai/qwen25vl3b_merged_0705'
!rm -f /content/outputs/raw_train500.jsonl
!cd /content/code && python -m src.infer.predict --model {OLD} --split train --fold train \
    --limit 500 --tta 1 --batch 16 --style mid --out /content/outputs/raw_train500.jsonl
import sys, json; sys.path.insert(0, '/content/code')
from src.infer.aggregate import aggregate_file
from src.eval.em import evaluate
from src.data.loader import load_split
preds, _ = aggregate_file('/content/outputs/raw_train500.jsonl', disperse_gate=False)
truth = load_split('train')
truth = truth[truth['Id'].isin(preds)]
print(json.dumps(evaluate(preds, truth), indent=2))

In [ ]:
# G1a) [G1] val 생성 TTA 4→8 확장 — 기존 raw에 신규 4뷰만 추가 생성 (재개 로직·perm 접두어 검증됨)
# 근거: track_b_0707.md — 합의도는 잘 보정된 지표(4/4 EM 0.80), 저합의 39%가 병목.
# 판정: val EM이 0.5173 대비 +1pp 이상이면 G1b(test) 진행.
MODEL = '/content/drive/MyDrive/snuai/qwen25vl7b_merged_0706'
!cd /content/code && python -m src.infer.predict --model {MODEL} --split train --fold val \
    --style mid --tta 8 --batch 16 --out /content/drive/MyDrive/snuai/raw_val_0706.jsonl
!cd /content/code && python -m src.infer.aggregate --raw /content/drive/MyDrive/snuai/raw_val_0706.jsonl \
    --out /content/outputs/pred_val_tta8.csv
!cd /content/code && python -m src.eval.em --pred /content/outputs/pred_val_tta8.csv --fold val

In [ ]:
# G1b) [G1] test TTA 8 확장 → 검증된 submission (제출 #1 후보)
!cd /content/code && python -m src.infer.predict --model {MODEL} --split test \
    --style mid --tta 8 --batch 16 --out /content/drive/MyDrive/snuai/raw_test_0706.jsonl
!cd /content/code && python -m src.infer.aggregate --raw /content/drive/MyDrive/snuai/raw_test_0706.jsonl \
    --submission /content/drive/MyDrive/snuai/submission_0707_tta8.csv
print('Drive의 snuai/submission_0707_tta8.csv 제출 (#1) — G1a가 +1pp 이상일 때만')

In [ ]:
# B1) 스모크 (32샘플) — 장기 학습 전 필수. 7B는 A100 전용 (T4 OOM)
import sys
sys.path.insert(0, '/content/code')  # 재실행 대비
from cloud.train_unsloth import run
SFT = '/content/code/outputs/sft_train.jsonl'
OUT = '/content/drive/MyDrive/snuai/outputs/qwen25vl7b_0707'  # Drive = 세션 휘발 대비
run('/content/code/configs/sft_qwen.yaml', SFT, DATA_ROOT, limit=32, output_dir=OUT + '_smoke')

In [ ]:
# B2) 본 학습 (_0707: 오버샘플 x2 + 3 epochs — configs/sft_qwen.yaml) — 유효 데이터 ~11k, _0706 대비 ~1.9x 시간
# 첫 실행 resume=False, 세션 끊긴 뒤 재실행은 resume=True
# 7B 배치 (유효 배치 16 고정, 3B _0705와 비교 가능성 유지):
#   A100 40GB -> {'per_device_batch': 2, 'grad_accum': 8}  (OOM 없고 여유 크면 4x4 시도)
#   L4 24GB   -> {'per_device_batch': 1, 'grad_accum': 16}
OVERRIDES = {'per_device_batch': 2, 'grad_accum': 8}  # A100 기준
lora_dir = run('/content/code/configs/sft_qwen.yaml', SFT, DATA_ROOT, resume=False,
               output_dir=OUT, train_overrides=OVERRIDES)
print('LoRA saved:', lora_dir)

In [ ]:
# B3) 병합 저장 (Drive) — 추론은 플레인 transformers 경로 사용
# ⚠️ save_pretrained_merged는 vision 모델에서 LoRA를 병합하지 않고 베이스만 저장
#    (unsloth#1352, 2026-07-05 실측: val EM이 랜덤으로 추락) → peft 표준 병합 사용.
from unsloth import FastVisionModel
OUT = '/content/drive/MyDrive/snuai/outputs/qwen25vl7b_0707'
MERGED = '/content/drive/MyDrive/snuai/qwen25vl7b_merged_0707'
model, processor = FastVisionModel.from_pretrained(OUT + '/lora', load_in_4bit=False)
merged = model.merge_and_unload()
merged.save_pretrained(MERGED)
processor.save_pretrained(MERGED)
print('merged saved:', MERGED, '— 반드시 B4 병합 검증을 통과한 뒤에만 추론에 사용')

In [ ]:
# B4) 병합 검증 게이트 (필수) — train 8샘플 생성으로 LoRA가 실제 병합됐는지 확인
# 병합 실패 시(베이스만 저장됨) 정답률이 랜덤(~0/8)으로 나온다. 0705 정상 기준: 4/8 이상.
MERGED = '/content/drive/MyDrive/snuai/qwen25vl7b_merged_0707'
!rm -f /content/outputs/merge_check.jsonl
!cd /content/code && python -m src.infer.predict --model {MERGED} --split train --fold train \
    --limit 8 --tta 1 --batch 8 --style mid --out /content/outputs/merge_check.jsonl
import sys, json; sys.path.insert(0, '/content/code')
from src.utils.permutation import parse_permutation, unshuffle_rank_label, parse_answer_column
from src.data.loader import load_split
truth = load_split('train').set_index('Id')
hits = n = 0
for r in map(json.loads, open('/content/outputs/merge_check.jsonl')):
    rank = parse_permutation(r['text']); n += 1
    hits += rank is not None and unshuffle_rank_label(rank, r['perm']) == parse_answer_column(truth.loc[r['Id'], 'Answer'])
print(f'merge check: {hits}/{n} 일치')
assert n >= 8 and hits >= 4, '병합 실패 의심 (unsloth#1352) — B3 재확인 전 추론 진행 금지'
print('병합 검증 통과 — C1 진행')

In [ ]:
# C1) val 리허설 (제출 전 필수) — raw는 Drive에 직접 저장 (세션 휘발 대비)
# 판정 기준: 7B _0706의 val EM 0.5173 (G1a에서 tta8 값이 나왔으면 그 값). 이보다 낮으면 제출 중단하고 원인 규명.
MODEL = '/content/drive/MyDrive/snuai/qwen25vl7b_merged_0707'
!cd /content/code && python -m src.infer.predict --model $MODEL --split train --fold val \
    --style mid --tta 8 --batch 16 --out /content/drive/MyDrive/snuai/raw_val_0707.jsonl
!cd /content/code && python -m src.infer.aggregate --raw /content/drive/MyDrive/snuai/raw_val_0707.jsonl \
    --out /content/outputs/pred_val.csv
!cd /content/code && python -m src.eval.em --pred /content/outputs/pred_val.csv --fold val

In [ ]:
# C2) test 추론 → 검증된 submission (제출 #2: 재학습 7B + 투표 집계) — raw는 Drive에 직접 저장
# 손상 이미지로 중단되면 데이터 재압축해제 후 같은 명령 재실행 (완료분 자동 스킵).
!cd /content/code && python -m src.infer.predict --model $MODEL --split test \
    --style mid --tta 8 --batch 16 --out /content/drive/MyDrive/snuai/raw_test_0707.jsonl
!cd /content/code && python -m src.infer.aggregate --raw /content/drive/MyDrive/snuai/raw_test_0707.jsonl \
    --submission /content/drive/MyDrive/snuai/submission_0707.csv
print('Drive의 snuai/submission_0707.csv를 다운로드해 Kaggle에 제출 (#1)')

In [ ]:
# E1b) [G3·선택] LL 뷰 1→4 확장 — E1(tta1)은 투표에 전면 열세였음 (0.4753 vs 0.5173, track_b_0707.md)
# 뷰 4개 합산이면 대등해질 가능성만 검증. 기존 ll_val_0706.jsonl에 3뷰만 추가 (재개 로직).
# 완료 후 ll_val_0706.jsonl을 로컬 outputs/로 복사해 fuse sweep 재실행:
#   python -m src.infer.fuse sweep --raw outputs/raw_val_0706.jsonl --ll outputs/ll_val_0706.jsonl
# vote 대비 +2pp 이상일 때만 E2 진행. (재학습 모델로 갈아탄 경우 파일명 _0707로 새로 스코어)
MODEL = '/content/drive/MyDrive/snuai/qwen25vl7b_merged_0706'
!cd /content/code && python -m src.infer.ll_score score --model {MODEL} --split train --fold val \
    --tta 4 --chunk 25 --out /content/drive/MyDrive/snuai/ll_val_0706.jsonl \
  && python -m src.infer.ll_score sweep --raw /content/drive/MyDrive/snuai/ll_val_0706.jsonl --fold val

In [ ]:
# E2) [G3·선택] LL/융합 test 제출 — E1b + 로컬 fuse sweep이 vote 대비 +2pp 이상일 때만.
# 이전 OOM은 픽셀 캡 silent no-op이 원인 — _0707 코드(cap_pixels)로 해소. 구 파일은 캡 미적용이므로 삭제 후 재스코어.
POLICY, K, MARGIN = 'h1', 3, None  # ← 로컬 fuse sweep 확정값 기입 (필수)
assert MARGIN is not None, '로컬 fuse sweep의 확정 정책/margin을 먼저 기입하세요'
!rm -f /content/drive/MyDrive/snuai/ll_test_0706.jsonl
!cd /content/code && python -m src.infer.ll_score score --model {MODEL} --split test \
    --tta 4 --chunk 25 --out /content/drive/MyDrive/snuai/ll_test_0706.jsonl
!cd /content/code && python -m src.infer.fuse decide --raw /content/drive/MyDrive/snuai/raw_test_0706.jsonl \
    --ll /content/drive/MyDrive/snuai/ll_test_0706.jsonl --policy {POLICY} --k {K} --margin {MARGIN} \
    --submission /content/drive/MyDrive/snuai/submission_0707_fuse.csv
print('Drive의 snuai/submission_0707_fuse.csv 제출')

In [ ]:
# G4) 3090 24h 예산 실측 (docs/rules.md §3.3 — 유일한 잔여 규정 검증 항목)
# A100 실측 x 보수계수 3.0 -> 3090 예상. verdict가 OK(<=20h)인지 reports/track_b_0707.md에 기록.
MODEL = '/content/drive/MyDrive/snuai/qwen25vl7b_merged_0706'  # 최종 채택 모델로 교체
!cd /content/code && python -m src.infer.budget_check --model {MODEL} --n 20 --tta 8 --ll-tta 0 --batch 16
# LL/융합 경로 채택 시에만 추가 실측:
# !cd /content/code && python -m src.infer.budget_check --model {MODEL} --n 10 --tta 0 --ll-tta 4 --chunk 25